In [1]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import newton
import matplotlib.pyplot as plt

# Modified Newtonian Dynamics

In [ ]:
G = 1.0          # Normalized units: GM* = 1, h = 1
e = 0.6
a0_values = [0.0, 0.01, 0.1]

def mond_acc(r, a0):
    """Exact MOND acceleration from notes Eq.(56)"""
    if r < 1e-12:
        r = 1e-12
    a_N = G / r**2
    return 0.5 * (a_N + np.sqrt(a_N**2 + 4 * a_N * a0))

def plot_mond_figure1():
    fig, axs = plt.subplots(1, 3, figsize=(15, 5), dpi=120)
    fig.suptitle('Figure 1: MOND Orbits (GM=1, h=1, e=0.6)', fontsize=16)
    
    for idx, a0 in enumerate(a0_values):
        # Initial conditions at pericenter
        r_peri = 1 / (1 + e)          # p = 1
        v_peri = np.sqrt(G * (1 + e) / r_peri)
        
        y0 = [r_peri, 0.0, 0.0, v_peri]
        
        def deriv(t, y):
            x, y_pos, vx, vy = y
            r = np.hypot(x, y_pos)
            g = mond_acc(r, a0)
            ax = -g * (x / r)
            ay = -g * (y_pos / r)
            return [vx, vy, ax, ay]
        
        # Integrate over many periods
        sol = solve_ivp(deriv, [0, 200], y0, method='DOP853',
                        rtol=1e-10, atol=1e-12, dense_output=True)
        
        t = np.linspace(0, 200, 8000)
        orbit = sol.sol(t)
        x, y = orbit[0], orbit[1]
        
        axs[idx].plot(x, y, 'r-', lw=1.2)
        axs[idx].plot(0, 0, 'k*', ms=12, label='Central mass')
        axs[idx].set_title(f'$a_0 = {a0}$')
        axs[idx].set_xlabel('x')
        axs[idx].set_ylabel('y')
        axs[idx].axis('equal')
        axs[idx].grid(True, alpha=0.3)
        axs[idx].legend()
    
    plt.tight_layout()
    plt.show()

# Run the figure
plot_mond_figure1()